# Experiment 10: Unsupervised Movie Clustering & Recommendation Model
## Dimensionality Reduction (TruncatedSVD) + K-Means Clustering
**Goal:** Group movies into semantic clusters based on synopsis, genres, keywords, and popularity metrics, and build a cluster-constrained recommendation engine.


In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score
from sklearn.metrics.pairwise import cosine_similarity


### 1. Load Processed Movies Data


In [ ]:
movies_df = pd.read_csv("../data/processed/movies_clean.csv")
print("Cleaned Movies Shape:", movies_df.shape)
movies_df.head(3)


### 2. Feature Engineering & Dimensionality Reduction
- Text features from combined metadata (`overview + genres + keywords`) converted via `TfidfVectorizer(max_features=5000)`.
- Reduced to 50 dense components with `TruncatedSVD`.
- Scaled numeric features (`vote_average`, `popularity`) concatenated with `StandardScaler`.


In [ ]:
# Load pre-fitted artifacts
svd = joblib.load("../models/svd.pkl")
scaler = joblib.load("../models/scaler.pkl")
cluster_model = joblib.load("../models/cluster_model.pkl")
combined_features = np.load("../models/combined_features.npy")

print("Combined Feature Matrix Shape:", combined_features.shape)
print("KMeans Number of Clusters:", cluster_model.n_clusters)


### 3. Elbow Method & Silhouette Score Analysis


In [ ]:
from IPython.display import Image
Image(filename="../models/elbow_plot.png")


### 4. Cluster Distribution & Top Movies Inspection


In [ ]:
cluster_summary = movies_df.groupby('cluster').agg(
    movie_count=('id', 'count'),
    avg_rating=('vote_average', 'mean'),
    top_titles=('title', lambda x: ', '.join(x.head(3)))
).reset_index()

cluster_summary


### 5. Recommendation Engine Verification
Given a target movie (e.g. *Inception* or *The Dark Knight*), the system:
1. Identifies the movie's cluster assignment.
2. Filters candidates within the same cluster.
3. Computes cosine similarity between the target feature vector and candidates.
4. Ranks recommendations using a blended score of similarity and vote average.


In [ ]:
def get_recommendations_demo(title, top_n=5):
    matches = movies_df[movies_df['title'].str.lower() == title.lower()]
    if matches.empty:
        return f"Movie '{title}' not found."
    
    target_idx = matches.index[0]
    target_cluster = movies_df.loc[target_idx, 'cluster']
    
    cluster_indices = movies_df[(movies_df['cluster'] == target_cluster) & (movies_df.index != target_idx)].index
    target_vec = combined_features[target_idx].reshape(1, -1)
    cand_vecs = combined_features[cluster_indices]
    
    sims = cosine_similarity(target_vec, cand_vecs)[0]
    
    results = movies_df.loc[cluster_indices].copy()
    results['similarity'] = np.round(sims, 4)
    results['blend_score'] = results['similarity'] * 0.7 + (results['vote_average'] / 10.0) * 0.3
    
    top = results.sort_values(by=['blend_score', 'vote_average'], ascending=[False, False]).head(top_n)
    return top[['title', 'genres_clean', 'vote_average', 'similarity', 'blend_score']]

get_recommendations_demo("The Dark Knight Rises")


In [ ]:
get_recommendations_demo("Avatar")


### 6. Summary of Clustering Results
- Silhouette Score: ~0.211 with Davies-Bouldin Index of 0.992.
- The 15 clusters group movies effectively by thematic tone and genre signatures.
- Cosine similarity refinement within the cluster prevents noisy out-of-cluster false positives.
